## Imports

In [1]:
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error

/Users/akhilendra.singh/Documents/Trip-Duration-Prediction/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_parquet("../data/green_tripdata_2026-01.parquet")

## Global Variables

In [3]:
TRACKING_URI = "http://127.0.0.1:5000"

## Exploring the Data

In [4]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,...,1.5,7.5,0.0,NaN,1.0,45.20,1.0,1.0,2.75,0.75
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,...,0.0,10.2,0.0,NaN,1.0,61.20,1.0,2.0,0.00,0.00
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,...,1.5,2.0,0.0,NaN,1.0,46.00,1.0,1.0,0.00,0.00
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,...,1.5,0.0,0.0,NaN,1.0,25.05,2.0,1.0,2.75,0.00
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,...,0.5,0.0,0.0,NaN,1.0,21.60,2.0,1.0,0.00,0.00


In [5]:
df.shape

(40272, 21)

In [6]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge',
       'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee'],
      dtype='object')

In [7]:
df.isnull().sum()

VendorID                     0
lpep_pickup_datetime         0
lpep_dropoff_datetime        0
store_and_fwd_flag        5414
RatecodeID                5414
PULocationID                 0
DOLocationID                 0
passenger_count           5414
trip_distance                0
fare_amount                  0
extra                        0
mta_tax                      0
tip_amount                   0
tolls_amount                 0
ehail_fee                40272
improvement_surcharge        0
total_amount                 0
payment_type              5414
trip_type                 5415
congestion_surcharge      5414
cbd_congestion_fee           0
dtype: int64

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40272 entries, 0 to 40271
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   VendorID               40272 non-null  int32         
 1   lpep_pickup_datetime   40272 non-null  datetime64[us]
 2   lpep_dropoff_datetime  40272 non-null  datetime64[us]
 3   store_and_fwd_flag     34858 non-null  object        
 4   RatecodeID             34858 non-null  float64       
 5   PULocationID           40272 non-null  int32         
 6   DOLocationID           40272 non-null  int32         
 7   passenger_count        34858 non-null  float64       
 8   trip_distance          40272 non-null  float64       
 9   fare_amount            40272 non-null  float64       
 10  extra                  40272 non-null  float64       
 11  mta_tax                40272 non-null  float64       
 12  tip_amount             40272 non-null  float64       
 13  t

In [9]:
df.describe()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
count,40272.000000,40272,40272,34858.000000,40272.000000,40272.000000,34858.000000,40272.000000,40272.000000,40272.000000,40272.000000,40272.000000,40272.000000,0.0,40272.000000,40272.000000,34858.000000,34857.000000,34858.000000,40272.000000
mean,2.279897,2026-01-16 22:38:57.806366,2026-01-16 22:59:02.332389,1.207212,96.190480,141.878352,1.298382,12.551243,16.006925,0.823550,0.561687,2.476605,0.260190,NaN,0.922189,24.195461,1.259711,1.046562,0.851383,0.058179
min,1.000000,2025-12-27 16:49:41,2025-12-27 17:02:11,1.000000,1.000000,1.000000,0.000000,0.000000,-70.000000,-7.500000,-0.500000,-2.520000,0.000000,NaN,-1.000000,-76.500000,1.000000,1.000000,0.000000,0.000000
25%,2.000000,2026-01-09 11:18:24,2026-01-09 11:37:25.250000,1.000000,74.000000,74.000000,1.000000,1.200000,8.600000,0.000000,0.500000,0.000000,0.000000,NaN,1.000000,14.600000,1.000000,1.000000,0.000000,0.000000
50%,2.000000,2026-01-16 14:27:16.500000,2026-01-16 14:47:26.500000,1.000000,75.000000,140.000000,1.000000,1.960000,12.800000,0.000000,0.500000,2.000000,0.000000,NaN,1.000000,20.000000,1.000000,1.000000,0.000000,0.000000
75%,2.000000,2026-01-23 19:37:11.500000,2026-01-23 19:55:24.250000,1.000000,97.000000,229.000000,1.000000,3.470000,19.100000,1.000000,0.500000,3.770000,0.000000,NaN,1.000000,28.860000,1.000000,1.000000,2.750000,0.000000
max,6.000000,2026-02-01 21:08:36,2026-02-01 21:15:02,99.000000,265.000000,265.000000,9.000000,179830.920000,960.000000,7.500000,4.250000,300.000000,85.000000,NaN,1.000000,961.000000,4.000000,2.000000,2.750000,0.750000
std,1.233095,NaN,NaN,1.018101,55.683047,77.583150,0.950439,1033.875580,14.951959,1.332193,0.321908,3.564317,1.583427,NaN,0.245158,17.177547,0.471217,0.210701,1.271401,0.200626


In [10]:
df.drop('ehail_fee',axis=1,inplace=True)

## Creating the target variable

In [11]:
df = df.dropna()

In [12]:
df.shape

(34857, 20)

In [13]:
df['duration'] = ((df['lpep_dropoff_datetime']-df['lpep_pickup_datetime']).dt.total_seconds())/60

In [14]:
df['duration']# This is the duration in minutes.

0        27.300000
1        48.383333
2        21.300000
3        16.200000
4        18.600000
           ...    
34853     5.883333
34854     8.733333
34855     4.633333
34856     8.833333
34857    10.483333
Name: duration, Length: 34857, dtype: float64

In [15]:
#Since most of thr duration is below 60 minutes , hence keep it below 60 minutes
df = df[(df['duration']>=0) & (df['duration']<=60)]

In [16]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,duration
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,...,1.5,7.5,0.0,1.0,45.20,1.0,1.0,2.75,0.75,27.300000
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,...,0.0,10.2,0.0,1.0,61.20,1.0,2.0,0.00,0.00,48.383333
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,...,1.5,2.0,0.0,1.0,46.00,1.0,1.0,0.00,0.00,21.300000
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,...,1.5,0.0,0.0,1.0,25.05,2.0,1.0,2.75,0.00,16.200000
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,...,0.5,0.0,0.0,1.0,21.60,2.0,1.0,0.00,0.00,18.600000


In [17]:
df.shape

(34572, 21)

In [18]:
df = df[df['passenger_count']>0]

In [19]:
df.shape

(34014, 21)

## Feature Engineering

In [20]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee', 'duration'],
      dtype='object')

In [21]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee,duration
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,...,1.5,7.5,0.0,1.0,45.20,1.0,1.0,2.75,0.75,27.300000
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,...,0.0,10.2,0.0,1.0,61.20,1.0,2.0,0.00,0.00,48.383333
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,...,1.5,2.0,0.0,1.0,46.00,1.0,1.0,0.00,0.00,21.300000
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,...,1.5,0.0,0.0,1.0,25.05,2.0,1.0,2.75,0.00,16.200000
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,...,0.5,0.0,0.0,1.0,21.60,2.0,1.0,0.00,0.00,18.600000


In [22]:
df['pickup_hour'] = df['lpep_pickup_datetime'].dt.hour

In [23]:
df['pickup_day'] = df['lpep_pickup_datetime'].dt.day

In [24]:
df['pickup_month'] = df['lpep_pickup_datetime'].dt.month

In [25]:
df['pickup_weekday'] = df['lpep_pickup_datetime'].dt.weekday

In [26]:
df['PU_DO'] = ((df['PULocationID'].astype(str))+"_"+ (df['DOLocationID'].astype(str)))

In [27]:
df.columns

Index(['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime',
       'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID',
       'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax',
       'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount',
       'payment_type', 'trip_type', 'congestion_surcharge',
       'cbd_congestion_fee', 'duration', 'pickup_hour', 'pickup_day',
       'pickup_month', 'pickup_weekday', 'PU_DO'],
      dtype='object')

In [28]:
numerical_features = [

        # time features
        "pickup_hour",
        "pickup_day",
        "pickup_month",
        "pickup_weekday",
        "passenger_count",
        "trip_distance",
    ]

categorical_features = [
    "PULocationID",
    "DOLocationID",
    "PU_DO"
]

In [29]:
X = df[numerical_features+categorical_features]
y = df['duration']

In [30]:
X.head()

,pickup_hour,pickup_day,pickup_month,pickup_weekday,passenger_count,trip_distance,PULocationID,DOLocationID,PU_DO
0,0,1,1,3,2.0,6.20,65,233,65_233
1,0,1,1,3,5.0,5.36,66,188,66_188
2,0,1,1,3,4.0,10.60,65,179,65_179
3,0,1,1,3,1.0,4.20,42,141,42_141
4,0,1,1,3,1.0,2.76,95,82,95_82


## Splitting the Data and doing Scaling

In [31]:
X_train,X_val,y_train,y_val = train_test_split(X,y,test_size=0.3,random_state=42)
X_test,X_val,y_test,y_val = train_test_split(X_val,y_val,test_size=0.5,random_state=42)

In [32]:
X_train.head()

,pickup_hour,pickup_day,pickup_month,pickup_weekday,passenger_count,trip_distance,PULocationID,DOLocationID,PU_DO
11202,15,11,1,6,1.0,2.87,74,239,74_239
15934,13,15,1,3,1.0,0.96,166,152,166_152
26636,10,24,1,5,1.0,1.72,74,75,74_75
31073,2,29,1,3,1.0,1.21,129,226,129_226
24904,22,22,1,3,1.0,8.17,95,16,95_16


In [35]:
check = X_train.iloc[0].to_dict()

In [37]:
check.keys()

dict_keys(['pickup_hour', 'pickup_day', 'pickup_month', 'pickup_weekday', 'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID', 'PU_DO'])

In [38]:
u = pd.DataFrame([{'pickup_hour': 15,
'pickup_day': 11,
'pickup_month': 1,
'pickup_weekday': 6,
'passenger_count': 1.0,
'trip_distance': 2.87,
'PULocationID': 74,
'DOLocationID': 239,
'PU_DO': '74_239'}])

In [39]:
u

,pickup_hour,pickup_day,pickup_month,pickup_weekday,passenger_count,trip_distance,PULocationID,DOLocationID,PU_DO
0,15,11,1,6,1.0,2.87,74,239,74_239


In [42]:
list(u.keys())

['pickup_hour',
 'pickup_day',
 'pickup_month',
 'pickup_weekday',
 'passenger_count',
 'trip_distance',
 'PULocationID',
 'DOLocationID',
 'PU_DO']

In [35]:
y_train.head()

11202    19.566667
15934     7.866667
26636     9.816667
31073     6.383333
24904    15.933333
Name: duration, dtype: float64

In [37]:
reference_data = pd.concat([X_train,y_train],axis=1)

In [39]:
reference_data.to_csv("../monitoring/data/reference_data.csv")

In [33]:
y_train.head()

11202    19.566667
15934     7.866667
26636     9.816667
31073     6.383333
24904    15.933333
Name: duration, dtype: float64

In [33]:
# import numpy as np

# print("Positive infinity:", np.isposinf(X_train).sum().sum())
# print("Negative infinity:", np.isneginf(X_train).sum().sum())

In [34]:
# inf_columns = X_train.columns[
#     np.isinf(X_train.select_dtypes(include=np.number)).any()
# ]

# print(inf_columns)

In [35]:
column_transformer = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

## Model building and Experiment tracking

In [36]:
mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment("trip-duration-prediction")

<Experiment: artifact_location='/Users/akhilendra.singh/Documents/Trip-Duration-Prediction/model-building/artifacts/1', creation_time=1789375472721, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789375472721, lifecycle_stage='active', name='trip-duration-prediction', tags={}, trace_location=None, workspace='default'>

In [37]:
def get_params(model_name, trial):
    if model_name == 'Linear Regression':
        return {}
        
    elif model_name == 'SVR':  
        return {
            'C': trial.suggest_float('C', 1e-3, 1e2, log=True),
            'epsilon': trial.suggest_float('epsilon', 1e-4, 1.0, log=True),
            'kernel': trial.suggest_categorical('kernel', ['rbf', 'linear', 'poly']),
        }
        
    elif model_name == 'Decision Tree':
        return {
            'max_depth': trial.suggest_int('max_depth', 3, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        }
        
    elif model_name == 'Random Forest':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 100, 400, step=50),
            'max_depth': trial.suggest_int('max_depth', 3, 20),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        }
        
    elif model_name == 'Gradient Boost':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 50, 300, step=50),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
        }
        
    else:
        raise ValueError(f"Unknown model name: {model_name}")


In [38]:
y_val.head()

9003      9.350000
25646    30.516667
1487     14.100000
7405     29.400000
19968    40.350000
Name: duration, dtype: float64

In [39]:
def get_model(model_name,params):
    if model_name == 'Linear Regression':
        model = LinearRegression(**params)
            
    elif model_name == 'SVR':  
        model = SVR(**params)
     
    elif model_name == 'Decision Tree':  
        model = DecisionTreeRegressor(**params)

    elif model_name == 'Random Forest':  
        model = RandomForestRegressor(**params)

    elif model_name == 'Gradient Boost':  
        model = GradientBoostingRegressor(**params)
    else:
        raise ValueError(f"Unknown model name: {model_name}")

    return model
    

In [40]:
def train_model(model,n_trials):

    def objective(trial):
        params = get_params(model,trial)
        classifier = get_model(model,params)
        pipeline = Pipeline([
        ('preprocessor',column_transformer),
        ('model',classifier)
        ])
        with mlflow.start_run(nested=True):
            mlflow.log_params(params)
            pipeline.fit(X_train,y_train)
            y_pred = pipeline.predict(X_val)
            rmse = root_mean_squared_error(y_val,y_pred)
            mlflow.log_metric("rmse",rmse)

        return rmse
    
    with mlflow.start_run(run_name=f"{model}"):
        mlflow.log_param('model',model)
        study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
        study.optimize(objective, n_trials)

        mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
        mlflow.log_metric("best-rmse",study.best_value)

In [41]:
models = ['Linear Regression','SVR','Decision Tree','Random Forest','Gradient Boost']

for model in models:
    n_trials = 10 if model in ['Random Forest','Gradient Boost','SVR'] else 20
    train_model(model,n_trials)

[I 2026-09-15 11:04:12,030] A new study created in memory with name: no-name-5db44d2a-ef83-42b1-b5af-7a010c13d101
[I 2026-09-15 11:04:12,186] Trial 0 finished with value: 5.297774718297283 and parameters: {}. Best is trial 0 with value: 5.297774718297283.
[I 2026-09-15 11:04:12,319] Trial 1 finished with value: 5.297774718297283 and parameters: {}. Best is trial 0 with value: 5.297774718297283.


🏃 View run able-mink-607 at: http://127.0.0.1:5000/#/experiments/1/runs/86265345d8e044d787269df93bda9385
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run trusting-bear-693 at: http://127.0.0.1:5000/#/experiments/1/runs/c58292c1941c458b89952a9ed832f290
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-09-15 11:04:12,450] Trial 2 finished with value: 5.297774718297283 and parameters: {}. Best is trial 0 with value: 5.297774718297283.
[I 2026-09-15 11:04:12,579] Trial 3 finished with value: 5.297774718297283 and parameters: {}. Best is trial 0 with value: 5.297774718297283.


🏃 View run blushing-grouse-115 at: http://127.0.0.1:5000/#/experiments/1/runs/3ce593b9f1074b5e9045605a121f30c9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run marvelous-wren-490 at: http://127.0.0.1:5000/#/experiments/1/runs/79a8210e44cc414d9abfac0936b9c7e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-09-15 11:04:12,711] Trial 4 finished with value: 5.297774718297283 and parameters: {}. Best is trial 0 with value: 5.297774718297283.
[I 2026-09-15 11:04:12,842] Trial 5 finished with value: 5.297774718297283 and parameters: {}. Best is trial 0 with value: 5.297774718297283.


...

🏃 View run honorable-hare-984 at: http://127.0.0.1:5000/#/experiments/1/runs/a40235cb347f425096f93ef9b2217989
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-09-15 11:12:47,885] Trial 8 finished with value: 5.626429628212009 and parameters: {'n_estimators': 50, 'learning_rate': 0.022065329563125038, 'max_depth': 5}. Best is trial 3 with value: 4.3825722608797175.


🏃 View run adaptable-mouse-83 at: http://127.0.0.1:5000/#/experiments/1/runs/4bb94fee05fa411885ec14799b090e12
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


[I 2026-09-15 11:12:53,131] Trial 9 finished with value: 4.4170412775591865 and parameters: {'n_estimators': 100, 'learning_rate': 0.12158752897954846, 'max_depth': 10}. Best is trial 3 with value: 4.3825722608797175.


🏃 View run omniscient-carp-91 at: http://127.0.0.1:5000/#/experiments/1/runs/9739506d4b1a4e3fbaf541ed1a6fc8d8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run Gradient Boost at: http://127.0.0.1:5000/#/experiments/1/runs/935053fa8caa4fceb5f2754335498e89
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Model Registery

In [40]:
from mlflow import MlflowClient
client = MlflowClient(tracking_uri=TRACKING_URI)

In [43]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids=["1"],
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=5,
    order_by=["metrics.`best-rmse` ASC"]
)

## Taking out the best runs according to lowest RMSE

In [41]:
def get_best_params(model_name,params):
    if model_name == 'Linear Regression':
        return {}
            
    elif model_name == 'SVR':  
        return {
            'C': float(params["best_C"]),
            'epsilon': float(params["best_epsilon"]),
            'kernel': params["best_kernel"]
        }
        
    elif model_name == 'Decision Tree':
        return {
            'max_depth': int(params["best_max_depth"]),
            'min_samples_split': int(params["best_min_samples_split"]),
            'min_samples_leaf': int(params["best_min_samples_leaf"]),
        }
        
    elif model_name == 'Random Forest':
        return {
            'n_estimators': int(params["best_n_estimators"]),
            'max_depth': int(params["best_max_depth"]),
            'min_samples_split': int(params["best_min_samples_split"]),
            'min_samples_leaf': int(params["best_min_samples_leaf"]),
        }
        
    elif model_name == 'Gradient Boost':
        return {
            'n_estimators': int(params["best_n_estimators"]),
            'learning_rate':float(params["best_learning_rate"]),
            'max_depth': int(params["best_max_depth"]),
        }
        
    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [45]:

for run in runs:
    # print(run.data.tags.get("mlflow.runName"))
    # print(run.info)
    params = run.data.params
    # print(params)
    model = params["model"]
    best_params = get_best_params(model,params)
    print(model)
    print(best_params)
    print(run.data.metrics["best-rmse"])
    print("*"*50)

Gradient Boost
{'n_estimators': 250, 'learning_rate': 0.15291977365631307, 'max_depth': 9}
4.3825722608797175
**************************************************
Random Forest
{'n_estimators': 350, 'max_depth': 16, 'min_samples_split': 7, 'min_samples_leaf': 8}
4.546268886233306
**************************************************
SVR
{'C': 37.22622766508552, 'epsilon': 0.0035413396633088286, 'kernel': 'rbf'}
4.548722488367539
**************************************************
Decision Tree
{'max_depth': 14, 'min_samples_split': 7, 'min_samples_leaf': 20}
4.806084470053386
**************************************************
Linear Regression
{}
5.297774718297283
**************************************************


In [48]:
for run in runs:
    params = run.data.params
    model = params["model"]
    best_params = get_best_params(model,params)
    classifier = get_model(model,best_params)
    pipeline = Pipeline([
            ('preprocessor',column_transformer),
            ('model',classifier)
            ])
    with mlflow.start_run(nested=True):
        mlflow.log_params(best_params)
        mlflow.log_param("model",model)
        pipeline.fit(X_train,y_train)
        y_pred = pipeline.predict(X_val)
        rmse = root_mean_squared_error(y_val,y_pred)
        mlflow.log_metric("rmse",rmse)
        mlflow.sklearn.log_model(pipeline,artifact_path="model",skops_trusted_types=["scipy.sparse._csr.csr_matrix"])

2026/09/15 14:03:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/15 14:03:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run exultant-fowl-943 at: http://127.0.0.1:5000/#/experiments/1/runs/1fb82dbf423340f1aa1db2c7815ca8c6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/09/15 14:04:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/15 14:04:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run illustrious-goat-202 at: http://127.0.0.1:5000/#/experiments/1/runs/7f6bcc51d0a846c7bdc3c8c56e0752d5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/09/15 14:06:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/15 14:06:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/09/15 14:06:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run nervous-gnat-242 at: http://127.0.0.1:5000/#/experiments/1/runs/d22c01c2ccac44b38f0beb4923ec8b4a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/09/15 14:06:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/09/15 14:06:27 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run shivering-grouse-54 at: http://127.0.0.1:5000/#/experiments/1/runs/22bcaf6e8cb74dbf9a1a8551f11cf453
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/09/15 14:06:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run defiant-moose-258 at: http://127.0.0.1:5000/#/experiments/1/runs/e58d51a2a41d49a7b2066cd062bc7d6a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Registering the top 5 models and making one model champion

In [52]:
registered_model_name = "trip-duration-prediction"

In [49]:
runs = client.search_runs(
    experiment_ids=["1"],
    max_results=5,
    order_by=["start_time DESC"]
)

In [53]:
for run in runs:
    run_id = run.info.run_id
    model_uri = f"runs:/{run_id}/model"
    mlflow.register_model(model_uri=model_uri,name = registered_model_name)

Successfully registered model 'trip-duration-prediction'.
2026/09/15 14:15:16 WARNING mlflow.tracking._model_registry.fluent: Run with id e58d51a2a41d49a7b2066cd062bc7d6a has no artifacts at artifact path 'model', registering model based on models:/m-bc0850cca7844bfc80f355f5fc975098 instead
2026/09/15 14:15:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: trip-duration-prediction, version 1
Created version '1' of model 'trip-duration-prediction'.
Registered model 'trip-duration-prediction' already exists. Creating a new version of this model...
2026/09/15 14:15:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 22bcaf6e8cb74dbf9a1a8551f11cf453 has no artifacts at artifact path 'model', registering model based on models:/m-b0b935ed87584a6189cede7f708648cd instead
2026/09/15 14:15:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Mod

In [54]:
registered_models = client.search_model_versions(f"name='{registered_model_name}'")
len(registered_models)

5

In [60]:
for registered_model in registered_models:
    run_id = registered_model.run_id
    info = client.get_run(run_id)
    print(info.data.metrics["rmse"])
    print(registered_model.version)

4.383813979383911
5
4.547593586620477
4
4.548722488367539
3
4.806084470053386
2
5.297774718297283
1


In [61]:
best_rmse = 100
best_version = -1
for registered_model in registered_models:
    run_id = registered_model.run_id
    info = client.get_run(run_id)
    rmse = info.data.metrics["rmse"]

    if rmse<best_rmse:
        best_rmse = rmse
        best_version = registered_model.version

In [65]:
client.set_registered_model_alias(name = registered_model_name,alias="Champion",version=best_version)

In [66]:
model_uri = f"models:/{registered_model_name}@Champion"
model = mlflow.sklearn.load_model(model_uri=model_uri)

In [69]:
import joblib

joblib.dump(
    model,
    "./best_model.joblib"
)

['./best_model.joblib']

In [68]:
model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['pickup_hour','pickup_day','pickup_month',...,'PULocationID', 'DOLocationID','PU_DO']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subs

In [70]:
X_test.to_csv('../data/test_data.csv')

In [71]:
y_test.to_csv("../data/test_data_duration.csv")

In [44]:
import joblib
model = joblib.load("./best_model.joblib")

In [50]:
u

,pickup_hour,pickup_day,pickup_month,pickup_weekday,passenger_count,trip_distance,PULocationID,DOLocationID,PU_DO
0,15,11,1,6,1.0,2.87,74,239,74_239


In [62]:
z = pd.concat([u,u])

In [63]:
z

,pickup_hour,pickup_day,pickup_month,pickup_weekday,passenger_count,trip_distance,PULocationID,DOLocationID,PU_DO
0,15,11,1,6,1.0,2.87,74,239,74_239
0,15,11,1,6,1.0,2.87,74,239,74_239


In [64]:
i = model.named_steps['preprocessor'].transform(z)

In [65]:
i.shape

(2, 2788)

In [57]:
X['PULocationID'].nunique()

163

In [68]:
pred = model.predict(z)

In [69]:
pred

array([15.75473538, 15.75473538])

In [71]:
pd.isna(z).any()

pickup_hour        False
pickup_day         False
pickup_month       False
pickup_weekday     False
passenger_count    False
trip_distance      False
PULocationID       False
DOLocationID       False
PU_DO              False
dtype: bool